# CICCADA Stage 2 conformance-table builders

This notebook builds the flex-included Stage 2 sensitivity tables:

- `conformance_voltwatt_v2_flex_included`
- `conformance_voltwattghi_v2_flex_included`
- `conformance_voltvar_v2_flex_included`

Inputs:

- raw `ts` telemetry;
- existing `meta_up23c` metadata;
- `all_uncurtailedpv_v2_flex_included` from Stage 1.

Methodological settings:

- flexible-export-detected sites included;
- average site voltage used;
- `ac_capacity_kw` used as the standards rating proxy;
- `s_99` used for the empirical Volt-VAr apparent-limit symptom;
- `review_corrected` Volt-VAr capability profile selected;
- 2024 and 2025 processed using AEST month/day boundaries.

Run the smoke test first. After it passes, recreate the custom tables to remove
the smoke-test rows before starting the production run.

## 0. Setup

In [1]:
import sys
import time
import importlib
from pathlib import Path

# This assumes the notebook is running from:
# bms_sa_review/data_calc_write/stage2_conformance/
ROOT = Path.cwd().parents[1]

sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "shared"))
sys.path.insert(
    0,
    str(ROOT / "data_calc_write" / "stage2_conformance"),
)

assert (ROOT / "shared" / "aws_config.py").exists(), (
    f"Unexpected notebook working directory: {Path.cwd()}"
)

from aws_config import aq, tables
from ciccada_config import SAI
from stage2_common import aest_month_window

import as4777_curves
import build_conformance_voltvar as vv
import build_conformance_voltwatt as vw

# Reload after any local builder changes.
as4777_curves = importlib.reload(as4777_curves)
vv = importlib.reload(vv)
vw = importlib.reload(vw)

DB = SAI
N_PARTS = 8

Using VVAR_V3 = 240.0 V (AS4777.2:2020 Australia A)
Using VW_V1 = 253.0 V (AS4777.2:2020 Australia A)
Using QCAP_P_MIN = 20% of rated apparent power (capacity proxy selected and labelled at run time)
Using VW_V1 = 253.0 V (AS4777.2:2020 Australia A)
Using VVAR_V3 = 240.0 V (AS4777.2:2020 Australia A)
Using VW_V1 = 253.0 V (AS4777.2:2020 Australia A)
Using QCAP_P_MIN = 20% of rated apparent power (capacity proxy selected and labelled at run time)
Using VW_V1 = 253.0 V (AS4777.2:2020 Australia A)


In [2]:
# Stage 1 result consumed by GHI-aware Volt-Watt and Volt-VAr.
UNCURTAILED = "all_uncurtailedpv_v2_flex_included"

# Target tables for this run:
VW_BASIC_TARGET = "conformance_voltwatt_v2_flex_included"
VW_GHI_TARGET = "conformance_voltwattghi_v2_flex_included"
VV_TARGET = "conformance_voltvar_v2_flex_included"

FLEX_SELECTION = "include"

VW_OPTIONS = dict(
    rating_basis="ac_capacity_kw",
    voltage_aggregation="avg",
    flex_selection=FLEX_SELECTION,
)

VV_OPTIONS = dict(
    rating_basis="ac_capacity_kw",
    empirical_limit_basis="s_99",
    # Choose one from hossein_m3, review_corrected:
    capability_profile="review_corrected",
    voltage_aggregation="avg",
    flex_selection=FLEX_SELECTION,
)

print("Database:", DB)
print("Stage 1 counterfactual:", UNCURTAILED)
print("Volt-Watt target:", VW_BASIC_TARGET)
print("Volt-Watt GHI target:", VW_GHI_TARGET)
print("Volt-VAr target:", VV_TARGET)

Database: solar_analytics_iceberg
Stage 1 counterfactual: all_uncurtailedpv_v2_flex_included
Volt-Watt target: conformance_voltwatt_v2_flex_included
Volt-Watt GHI target: conformance_voltwattghi_v2_flex_included
Volt-VAr target: conformance_voltvar_v2_flex_included


In [3]:
# Connection + Stage 1 dependency check.
# Iceberg tables return nothing from DESCRIBE -- use SELECT * LIMIT 1.
stage1_dependency = aq(f"""
    SELECT
        count(*) AS n_rows,
        count(DISTINCT site_id) AS n_sites,
        min(t_stamp) AS first_t_stamp,
        max(t_stamp) AS last_t_stamp
    FROM {UNCURTAILED}
""", database=DB)

display(stage1_dependency)

assert int(stage1_dependency["n_rows"].iloc[0]) > 0
assert int(stage1_dependency["n_sites"].iloc[0]) > 0

print("Stage 1 counterfactual dependency exists.")

,n_rows,n_sites,first_t_stamp,last_t_stamp
0,504416916,15888,2024-01-01,2025-12-31 23:55:00


Stage 1 counterfactual dependency exists.


In [4]:
stage1_keys = aq(f"""
    WITH key_counts AS (
        SELECT
            site_id,
            t_stamp,
            count(*) AS n
        FROM {UNCURTAILED}
        GROUP BY site_id, t_stamp
    )
    SELECT
        count_if(n > 1) AS duplicated_keys,
        coalesce(
            sum(CASE WHEN n > 1 THEN n - 1 ELSE 0 END),
            0
        ) AS excess_rows,
        max(n) AS maximum_rows_per_key
    FROM key_counts
""", database=DB)

display(stage1_keys)

duplicated_keys = int(stage1_keys["duplicated_keys"].iloc[0])
excess_rows = int(stage1_keys["excess_rows"].iloc[0])
maximum_rows = int(stage1_keys["maximum_rows_per_key"].iloc[0])

assert duplicated_keys == 0
assert excess_rows == 0
assert maximum_rows <= 1

print("Stage 1 uniqueness check passed.")

,duplicated_keys,excess_rows,maximum_rows_per_key
0,0,0,1


Stage 1 uniqueness check passed.


In [5]:
assert FLEX_SELECTION == "include"
assert VW_OPTIONS["flex_selection"] == "include"
assert VV_OPTIONS["flex_selection"] == "include"

assert VW_OPTIONS["rating_basis"] == "ac_capacity_kw"
assert VV_OPTIONS["rating_basis"] == "ac_capacity_kw"
assert VV_OPTIONS["empirical_limit_basis"] == "s_99"

assert VW_OPTIONS["voltage_aggregation"] == "avg"
assert VV_OPTIONS["voltage_aggregation"] == "avg"

assert VV_OPTIONS["capability_profile"] in {
    "review_corrected",
    "hossein_m3",
}

assert all(
    target.endswith("_v2_flex_included")
    for target in (
        VW_BASIC_TARGET,
        VW_GHI_TARGET,
        VV_TARGET,
    )
)

print("Stage 2 run configuration passed.")

Stage 2 run configuration passed.


In [6]:
# Stage 2 metadata cardinality check.
#
# Stage 2 joins raw telemetry to meta_up23c by circuit_id. Each eligible
# circuit must resolve to exactly one site, polarity, capacity and S_99.
metadata_diagnostics = aq("""
    WITH eligible_metadata AS (
        SELECT DISTINCT
            circuit_id,
            site_id,
            circuit_polarity,
            ac_capacity_kw,
            s_99
        FROM meta_up23c
        WHERE is_pv = true
          AND ac_capacity_kw > 0
          AND s_99 > 0
    ),
    per_circuit AS (
        SELECT
            circuit_id,
            count(*) AS n_variants,
            count(DISTINCT site_id) AS n_sites,
            count(DISTINCT circuit_polarity) AS n_polarities,
            count(DISTINCT ac_capacity_kw) AS n_capacities,
            count(DISTINCT s_99) AS n_s99_values,
            count_if(site_id IS NULL) AS null_site_rows,
            count_if(circuit_polarity IS NULL) AS null_polarity_rows
        FROM eligible_metadata
        GROUP BY circuit_id
    )
    SELECT
        count(*) AS eligible_circuits,
        count_if(n_variants > 1) AS circuits_with_variants,
        count_if(n_sites > 1) AS circuits_with_multiple_sites,
        count_if(n_polarities > 1) AS circuits_with_conflicting_polarity,
        count_if(n_capacities > 1) AS circuits_with_conflicting_capacity,
        count_if(n_s99_values > 1) AS circuits_with_conflicting_s99,
        count_if(null_site_rows > 0) AS circuits_with_null_site,
        count_if(null_polarity_rows > 0) AS circuits_with_null_polarity
    FROM per_circuit
""", database=DB)

display(metadata_diagnostics)

,eligible_circuits,circuits_with_variants,circuits_with_multiple_sites,circuits_with_conflicting_polarity,circuits_with_conflicting_capacity,circuits_with_conflicting_s99,circuits_with_null_site,circuits_with_null_polarity
0,27108,0,0,0,0,0,0,0


In [7]:
diag = metadata_diagnostics.iloc[0]

fatal_columns = [
    "circuits_with_multiple_sites",
    "circuits_with_conflicting_polarity",
    "circuits_with_conflicting_capacity",
    "circuits_with_conflicting_s99",
    "circuits_with_null_site",
    "circuits_with_null_polarity",
]

failures = {
    column: int(diag[column])
    for column in fatal_columns
    if int(diag[column]) != 0
}

assert not failures, f"Ambiguous Stage 2 circuit metadata: {failures}"

print("Stage 2 metadata cardinality check passed.")

Stage 2 metadata cardinality check passed.


In [8]:
stage1_flex_membership = aq(f"""
    WITH counterfactual_sites AS (
        SELECT DISTINCT site_id
        FROM {UNCURTAILED}
    ),

    metadata AS (
        SELECT
            site_id,
            bool_or(
                coalesce(flex_export_detected, False)
            ) AS flex_export_detected
        FROM meta_up23c
        WHERE is_pv = True
        GROUP BY site_id
    )

    SELECT
        count(*) AS counterfactual_sites_with_metadata,

        count_if(
            coalesce(m.flex_export_detected, False)
        ) AS flex_sites,

        count_if(
            NOT coalesce(m.flex_export_detected, False)
        ) AS non_flex_sites

    FROM counterfactual_sites c
    INNER JOIN metadata m
        ON c.site_id = m.site_id
""", database=DB)

display(stage1_flex_membership)

assert int(
    stage1_flex_membership["flex_sites"].iloc[0]
) > 0, (
    "The flex-included Stage 1 counterfactual contains no detected flex sites"
)

print("Stage 1 dependency includes detected flexible-export sites.")

,counterfactual_sites_with_metadata,flex_sites,non_flex_sites
0,15888,534,15354


Stage 1 dependency includes detected flexible-export sites.


## 1. Read the SQL before running it


`preview_sql` builds the exact INSERT for one slice without executing it.


Check the AEST window: for AEST January 2024 it should read UTC partitions
`(2023,12)` and `(2024,1)`, from `2023-12-31 14:00:00` to `2024-01-31 14:00:00`.

In [9]:
print(aest_month_window(2024, 1))   # -> ('2023-12-31 14:00:00', '2024-01-31 14:00:00', [(2023,12),(2024,1)])
print(aest_month_window(2024, 7))
print(aest_month_window(2025, 12))

('2023-12-31 14:00:00', '2024-01-31 14:00:00', [(2023, 12), (2024, 1)])
('2024-06-30 14:00:00', '2024-07-31 14:00:00', [(2024, 6), (2024, 7)])
('2025-11-30 14:00:00', '2025-12-31 14:00:00', [(2025, 11), (2025, 12)])


In [10]:
import math
import re

selected_profile = VV_OPTIONS["capability_profile"]


def sql_contains(sql, pattern):
    return re.search(
        pattern,
        sql,
        flags=re.IGNORECASE | re.DOTALL,
    ) is not None


# -------------------------------------------------------------------------
# Python curve checks
# -------------------------------------------------------------------------

S = 5.0

physical_checks = {
    0.0: 0.0,
    0.5: 0.0,
    1.0: -2.2,
    3.0: -2.2,
    3.5: -2.625,
    4.0: -3.0,
    4.5: -math.sqrt(4.75),
    5.0: 0.0,
    5.5: 0.0,
}

for p_kw, expected_q_kvar in physical_checks.items():
    actual_q_kvar = as4777_curves.q_cap_absorbing(p_kw, S)

    assert math.isclose(
        actual_q_kvar,
        expected_q_kvar,
        abs_tol=1e-12,
    ), (
        f"Physical capability failure at P={p_kw}: "
        f"expected {expected_q_kvar}, got {actual_q_kvar}"
    )


response_checks = {
    0.0: 0.0,
    0.5: 0.0,
    1.0: -2.2,
    3.0: -2.2,
    3.5: -2.625,
    4.0: -3.0,
    4.5: -3.0,
    5.0: -3.0,
    5.5: -3.0,
}

for p_kw, expected_q_kvar in response_checks.items():
    actual_q_kvar = (
        as4777_curves.q_conformance_floor_absorbing(
            p_kw,
            S,
        )
    )

    assert math.isclose(
        actual_q_kvar,
        expected_q_kvar,
        abs_tol=1e-12,
    ), (
        f"Conformance-floor failure at P={p_kw}: "
        f"expected {expected_q_kvar}, got {actual_q_kvar}"
    )

print("Python capability-curve tests passed.")


# -------------------------------------------------------------------------
# Generate the exact SQL used by each builder
# -------------------------------------------------------------------------

vv_preview = vv.preview_sql(
    year=2024,
    month=1,
    n_parts=N_PARTS,
    part=0,
    target=VV_TARGET,
    uncurtailed=UNCURTAILED,
    **VV_OPTIONS,
)

vw_basic_preview = vw.preview_sql(
    which="basic",
    year=2024,
    month=1,
    n_parts=N_PARTS,
    part=0,
    target=VW_BASIC_TARGET,
    **VW_OPTIONS,
)

vw_ghi_preview = vw.preview_sql(
    which="ghi",
    year=2024,
    month=1,
    n_parts=N_PARTS,
    part=0,
    target=VW_GHI_TARGET,
    uncurtailed=UNCURTAILED,
    **VW_OPTIONS,
)

# -------------------------------------------------------------------------
# Deterministic Volt-Watt exposure-boundary checks
# -------------------------------------------------------------------------

vw_exposure_pattern = (
    r"round\s*\(\s*V\s*,\s*6\s*\)"
    r"\s*>\s*253(?:\.0+)?"
)

basic_exposure_gates = re.findall(
    vw_exposure_pattern,
    vw_basic_preview,
    flags=re.IGNORECASE,
)

ghi_exposure_gates = re.findall(
    vw_exposure_pattern,
    vw_ghi_preview,
    flags=re.IGNORECASE,
)

assert len(basic_exposure_gates) == 2, (
    "Basic Volt-Watt SQL must use the stable exposure test twice; "
    f"found {len(basic_exposure_gates)}"
)

assert len(ghi_exposure_gates) == 6, (
    "GHI-aware Volt-Watt SQL must use the stable exposure test six times; "
    f"found {len(ghi_exposure_gates)}"
)

print("Deterministic Volt-Watt exposure checks passed.")

# -------------------------------------------------------------------------
# Target-table checks
# -------------------------------------------------------------------------

assert sql_contains(
    vv_preview,
    rf"INSERT\s+INTO\s+{re.escape(VV_TARGET)}\b",
), "Volt-VAr SQL writes to the wrong target"

assert sql_contains(
    vw_basic_preview,
    rf"INSERT\s+INTO\s+{re.escape(VW_BASIC_TARGET)}\b",
), "Basic Volt-Watt SQL writes to the wrong target"

assert sql_contains(
    vw_ghi_preview,
    rf"INSERT\s+INTO\s+{re.escape(VW_GHI_TARGET)}\b",
), "GHI-aware Volt-Watt SQL writes to the wrong target"

# -------------------------------------------------------------------------
# Q-impact nearest-edge regression checks
# -------------------------------------------------------------------------
nearest_edge_pattern = (
    r"abs\s*\(\s*Q_kvar\s*-\s*Q_max_final\s*\)"
    r"\s*<=\s*"
    r"abs\s*\(\s*Q_kvar\s*-\s*Q_min_final\s*\)"
)

assert sql_contains(
    vv_preview,
    nearest_edge_pattern,
), "Volt-VAr Q_impact does not select the nearest band edge"

old_broken_pattern = (
    r"abs\s*\(\s*Q_kvar\s*\)\s*/"
    r"\s*\(\s*abs\s*\(\s*Q_max_final\s*\)"
    r"\s*\+\s*1e-9\s*\)\s*<="
)

assert not sql_contains(
    vv_preview,
    old_broken_pattern,
), "Old Q_impact ratio-comparison bug is still present"

# -------------------------------------------------------------------------
# Flex-inclusion provenance checks
# -------------------------------------------------------------------------

for name, sql in {
    "Volt-VAr": vv_preview,
    "Volt-Watt basic": vw_basic_preview,
    "Volt-Watt GHI": vw_ghi_preview,
}.items():
    assert sql_contains(
        sql,
        r"'include'\s+AS\s+flex_selection\b",
    ), f"{name} SQL does not store flex_selection='include'"


# -------------------------------------------------------------------------
# Capacity and tolerance checks
# -------------------------------------------------------------------------

assert sql_contains(
    vv_preview,
    r"0\.44\s*\*\s*ac_capacity_kw",
), "Volt-VAr 44% requirement is missing"

assert sql_contains(
    vv_preview,
    r"0\.04\s*\*\s*ac_capacity_kw",
), "Volt-VAr 4% tolerance is missing"

assert sql_contains(
    vv_preview,
    r"AS\s+capability_assessable\b",
), "Volt-VAr assessability field is missing"


# -------------------------------------------------------------------------
# Capability-profile checks
# -------------------------------------------------------------------------

if selected_profile == "review_corrected":
    assert sql_contains(
        vv_preview,
        r"ELSE\s+-0\.6\s*\*\s*ac_capacity_kw",
    ), "Reviewed reactive-power-priority floor is missing"

    assert sql_contains(
        vv_preview,
        r"abs\s*\(\s*P_kW\s*\)\s*>=\s*"
        r"0\.2\s*\*\s*ac_capacity_kw",
    ), "Reviewed 20% assessability rule is missing"

elif selected_profile == "hossein_m3":
    assert sql_contains(
        vv_preview,
        r"power\s*\(\s*ac_capacity_kw\s*,\s*2\s*\)"
        r"\s*-\s*"
        r"power\s*\(\s*abs\s*\(\s*P_kW\s*\)\s*,\s*2\s*\)",
    ), "Hossein apparent-power-circle expression is missing"

    assert sql_contains(
        vv_preview,
        r"1\s+AS\s+capability_assessable\b",
    ), "Hossein profile should mark all intervals assessable"

else:
    raise ValueError(
        f"Unknown capability profile: {selected_profile!r}"
    )

print("All Stage 2 generated-SQL preflight tests passed.")
print("Selected Volt-VAr profile:", selected_profile)

Python capability-curve tests passed.
Deterministic Volt-Watt exposure checks passed.
All Stage 2 generated-SQL preflight tests passed.
Selected Volt-VAr profile: review_corrected


## 2. Create the empty tables

Destructive: drops and recreates. `_v2` suffix throughout.
(originals never touched)

In [ ]:
print(vw.create_table_basic(
    aq,
    database=DB,
    target=VW_BASIC_TARGET,
))

print(vw.create_table_ghi(
    aq,
    database=DB,
    target=VW_GHI_TARGET,
))

print(vv.create_table(
    aq,
    database=DB,
    target=VV_TARGET,
))

time.sleep(5)   
tables(DB)[tables(DB)["Table"].str.contains("conformance")][["Table"]]

## 3. Test one AEST month, one site slice

`parts=[0]` is 1/8 of sites for AEST January 2024. This should complete in a few minutes. 

In [ ]:
vv.run_months_voltvar(
    aq,
    database=DB,
    year=2024,
    months=[1],
    n_parts=N_PARTS,
    parts=[0],
    target=VV_TARGET,
    uncurtailed=UNCURTAILED,
    **VV_OPTIONS,
)

In [ ]:
vw.run_months_basic(
    aq,
    database=DB,
    year=2024,
    months=[1],
    n_parts=N_PARTS,
    parts=[0],
    target=VW_BASIC_TARGET,
    **VW_OPTIONS,
)

In [ ]:
vw.run_months_ghi(
    aq,
    database=DB,
    year=2024,
    months=[1],
    n_parts=N_PARTS,
    parts=[0],
    target=VW_GHI_TARGET,
    uncurtailed=UNCURTAILED,
    **VW_OPTIONS,
)

## 4. Validate the test

Every "MUST be 0" line must actually be 0 before you go any further.

The one to watch is **duplicate keys**. If that is non-zero, the AEST window logic has leaked and a site-day has been split across two INSERTs.

In [ ]:
vv_test = vv.validate(
    aq,
    database=DB,
    target=VV_TARGET,
)


In [ ]:

vw_basic_test = vw.validate_basic(
    aq,
    database=DB,
    target=VW_BASIC_TARGET,
)



In [ ]:
vw_ghi_test = vw.validate_ghi(
    aq,
    database=DB,
    target=VW_GHI_TARGET,
)

In [ ]:
# Eyeball actual rows. 
# Check: day spans 1..31, day_night has both values,
# and P_kW_sum is positive during the day.
aq(f"""
    SELECT site_id, year, month, day, day_night,
           round(P_kW_sum, 1)                   AS P_kW_sum,
           round(nonconformance_voltvar_sum, 3) AS nonconf,
           round(curtailment_voltvar_sum, 3)    AS curtail,
           curtailment_eligible_count, null_uncurtailed_P_count,
           exposed_count, all_intervals_count, total_count
    FROM {VV_TARGET}
    ORDER BY curtailment_voltvar_sum DESC NULLS LAST
    LIMIT 15
""", database=DB)

In [ ]:
# regression test: the AEST boundary.
# Under original UTC extraction, intervals from 00:00-09:55 AEST were booked to
# the previous day. Here, day 1 of the month must contain a full AEST day --
# including its early-morning (night) intervals, which live in the PREVIOUS UTC
# month's partition. If day=1 has far fewer intervals than day=2, the window
# logic is wrong.
# NOTE: Since the data starts on 2024, we are missing the 10h from 31-DEC-2023, so day=1 will have 10 fewer intervals than day=2.
aq(f"""
    SELECT day, sum(all_intervals_count) AS intervals, count(DISTINCT site_id) AS sites
    FROM {VV_TARGET}
    WHERE year = 2024 AND month = 1 AND day IN (1, 2, 15, 30, 31)
    GROUP BY day ORDER BY day
""", database=DB)

In [ ]:
smoke_test_provenance = aq(f"""
    SELECT
        'volt_watt_basic' AS result,
        count(*) AS n_rows,
        count(DISTINCT site_id) AS n_sites,
        min(year) AS first_year,
        max(year) AS last_year,
        min(month) AS first_month,
        max(month) AS last_month,
        max(flex_selection) AS flex_selection
    FROM {VW_BASIC_TARGET}

    UNION ALL

    SELECT
        'volt_watt_ghi',
        count(*),
        count(DISTINCT site_id),
        min(year),
        max(year),
        min(month),
        max(month),
        max(flex_selection)
    FROM {VW_GHI_TARGET}

    UNION ALL

    SELECT
        'volt_var',
        count(*),
        count(DISTINCT site_id),
        min(year),
        max(year),
        min(month),
        max(month),
        max(flex_selection)
    FROM {VV_TARGET}
""", database=DB)

display(smoke_test_provenance)

assert (smoke_test_provenance["n_rows"] > 0).all()
assert set(smoke_test_provenance["first_year"]) == {2024}
assert set(smoke_test_provenance["last_year"]) == {2024}
assert set(smoke_test_provenance["first_month"]) == {1}
assert set(smoke_test_provenance["last_month"]) == {1}
assert set(smoke_test_provenance["flex_selection"]) == {"include"}

print("Smoke-test rows are in the intended custom tables.")

## 5. Full load (2024 **and** 2025)

Each call is one AEST month * 8 site-slices = 8 Athena queries. 
A full year is 96 queries per table. 
Run one table at a time and check the printout.

If Athena throttles (`TooManyRequestsException`), 
drop `N_PARTS` to 4 or run `months` in two halves.

In [ ]:
# Destructive only for the three explicitly named flex-included targets.
# This removes the January part-0 smoke-test rows before the production load.

print(vw.create_table_basic(
    aq,
    database=DB,
    target=VW_BASIC_TARGET,
))

print(vw.create_table_ghi(
    aq,
    database=DB,
    target=VW_GHI_TARGET,
))

print(vv.create_table(
    aq,
    database=DB,
    target=VV_TARGET,
))

time.sleep(5)

empty_targets = aq(f"""
    SELECT
        '{VW_BASIC_TARGET}' AS table_name,
        count(*) AS n_rows
    FROM {VW_BASIC_TARGET}

    UNION ALL

    SELECT
        '{VW_GHI_TARGET}',
        count(*)
    FROM {VW_GHI_TARGET}

    UNION ALL

    SELECT
        '{VV_TARGET}',
        count(*)
    FROM {VV_TARGET}
""", database=DB)

display(empty_targets)

assert (empty_targets["n_rows"] == 0).all(), (
    "At least one production target still contains smoke-test rows"
)

print("Smoke-test rows cleared; production targets are empty.")

In [ ]:
MONTHS = list(range(1, 13))

for year in (2024, 2025):
    vw.run_months_basic(
        aq,
        database=DB,
        year=year,
        months=MONTHS,
        n_parts=N_PARTS,
        target=VW_BASIC_TARGET,
        **VW_OPTIONS,
    )

In [ ]:
for year in (2024, 2025):
    vw.run_months_ghi(
        aq,
        database=DB,
        year=year,
        months=MONTHS,
        n_parts=N_PARTS,
        target=VW_GHI_TARGET,
        uncurtailed=UNCURTAILED,
        **VW_OPTIONS,
    )

In [12]:
for year in (2024, 2025):
    vv.run_months_voltvar(
        aq,
        database=DB,
        year=year,
        months=MONTHS,
        n_parts=N_PARTS,
        target=VV_TARGET,
        uncurtailed=UNCURTAILED,
        **VV_OPTIONS,
    )

loaded AEST 2024-01 part=0/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=1/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=2/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=3/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=4/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=5/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=6/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=7/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-02 part=0/8 (UTC 2024-01-31 14:00:00 -> 2024-02-29 14:00:00, partitions

## 6. Full validation

In [ ]:
vw.validate_basic(
    aq,
    database=DB,
    target=VW_BASIC_TARGET,
)

vw.validate_ghi(
    aq,
    database=DB,
    target=VW_GHI_TARGET,
)

vv.validate(
    aq,
    database=DB,
    target=VV_TARGET,
)

vw.cross_check(
    aq,
    database=DB,
    target_basic=VW_BASIC_TARGET,
    target_ghi=VW_GHI_TARGET,
)

### 6.1. Volt-Var

In [14]:
vvar_interval_balance = aq(f"""
    SELECT
        year,
        month,
        count(*) AS n_rows,
        count(DISTINCT site_id) AS n_sites,

        sum(total_count) AS assessable_intervals,
        sum(low_power_count) AS low_power_intervals,
        sum(all_intervals_count) AS all_intervals,

        sum(total_count)
            + sum(low_power_count)
            - sum(all_intervals_count)
            AS interval_balance

    FROM {VV_TARGET}

    GROUP BY year, month
    ORDER BY year, month
""", database=DB)

display(vvar_interval_balance)

assert (
    vvar_interval_balance["interval_balance"].fillna(0) == 0
).all(), "Volt-VAr assessable/low-power interval balance failed"

assert len(vvar_interval_balance) == 24, (
    "Expected 24 monthly partitions covering 2024 and 2025"
)

print("Volt-VAr interval balance and monthly coverage passed.")

,year,month,n_rows,n_sites,assessable_intervals,low_power_intervals,all_intervals,interval_balance
0,2024,1,683786,11613,32471804,63752217,96224021,0
1,2024,2,657095,11824,30582898,63235409,93818307,0
2,2024,3,710293,11823,31506633,69964768,101471401,0
3,2024,4,693979,11942,26876736,72301510,99178246,0
4,2024,5,717706,12065,23742967,78722780,102465747,0
5,2024,6,687738,12013,20385825,77799354,98185179,0
6,2024,7,683188,11654,22168228,75281017,97449245,0
7,2024,8,679766,11434,25345813,71560700,96906513,0
8,2024,9,657806,11403,29453324,64499459,93952783,0
9,2024,10,681196,11480,32988822,64189830,97178652,0


Volt-VAr interval balance and monthly coverage passed.


### 6.2. Volt-Watt

In [ ]:
vw.validate_basic(
    aq,
    database=DB,
    target=VW_BASIC_TARGET,
)

In [ ]:
vw.validate_ghi(
    aq,
    database=DB,
    target=VW_GHI_TARGET,
)


In [ ]:
# R13 regression test: the two Volt-Watt tables must share a denominator.
vw.cross_check(
    aq,
    database=DB,
    target_basic=VW_BASIC_TARGET,
    target_ghi=VW_GHI_TARGET,
)

In [ ]:
vw_denominator_mismatches = aq(f"""
    SELECT
        count(*) AS mismatched_rows,
        max(abs(g.total_count - b.total_count))
            AS maximum_total_count_difference,
        max(abs(g.all_intervals_count - b.all_intervals_count))
            AS maximum_all_interval_difference

    FROM {VW_BASIC_TARGET} b

    INNER JOIN {VW_GHI_TARGET} g
        ON b.site_id = g.site_id
       AND b.year = g.year
       AND b.month = g.month
       AND b.day = g.day
       AND b.day_night = g.day_night

    WHERE b.total_count <> g.total_count
       OR b.all_intervals_count <> g.all_intervals_count
""", database=DB)

display(vw_denominator_mismatches)

assert int(
    vw_denominator_mismatches["mismatched_rows"].iloc[0]
) == 0, "Basic and GHI-aware Volt-Watt denominators differ"

print("Volt-Watt denominator reconciliation passed.")

In [ ]:
monthly_cf_coverage = aq(f"""
    SELECT
        year,
        month,
        sum(curtailment_eligible_count) AS eligible,
        sum(null_uncurtailed_P_count) AS missing_counterfactual,
        sum(curtailment_eligible_count)
            - sum(null_uncurtailed_P_count) AS usable_counterfactual,
        round(
            100.0 * (
                sum(curtailment_eligible_count)
                - sum(null_uncurtailed_P_count)
            )
            / nullif(sum(curtailment_eligible_count), 0),
            2
        ) AS usable_pct
    FROM {VV_TARGET}
    GROUP BY year, month
    ORDER BY year, month
""", database=DB)

monthly_cf_coverage

### 6 OTHER

In [13]:
shape, dupes, coherence, cover = vv.validate(
    aq,
    database=DB,
    target=VV_TARGET,
)

expected_months = {
    (year, month)
    for year in (2024, 2025)
    for month in range(1, 13)
}

actual_months = set(
    zip(
        shape["year"].astype(int),
        shape["month"].astype(int),
    )
)

assert actual_months == expected_months, (
    f"Expected 24 months, but found: {sorted(actual_months)}"
)

assert (shape["n_rows"] > 0).all(), (
    "At least one month has no results"
)

assert int(dupes["n_dupe_keys"].iloc[0]) == 0, (
    "Duplicate site/day/day-night records found"
)

assert (
    coherence.fillna(0).to_numpy() == 0
).all(), "A Volt-VAr coherence check failed"

print("Corrected Volt-VAr rebuild passed structural validation.")

Rows / sites / days per AEST month:
 year  month  n_rows  n_sites  n_days
 2024      1  683786    11613      31
 2024      2  657095    11824      29
 2024      3  710293    11823      31
 2024      4  693979    11942      30
 2024      5  717706    12065      31
 2024      6  687738    12013      30
 2024      7  683188    11654      31
 2024      8  679766    11434      31
 2024      9  657806    11403      30
 2024     10  681196    11480      31
 2024     11  664490    11578      30
 2024     12  696203    11599      31
 2025      1  698003    11802      31
 2025      2  621949    11564      28
 2025      3  659543    11202      31
 2025      4  639322    10977      30
 2025      5  654187    10974      31
 2025      6  539302    10692      30
 2025      7  645443    10912      31
 2025      8  634161    10602      31
 2025      9  599568    10423      30
 2025     10  606508    10213      31
 2025     11  570116     9905      30
 2025     12  572385     9620      31

Duplicate (ye

## 7. Reconcile against the original tables

Differences from the original tables are expected because:

- this build covers both 2024 and 2025;
- flexible-export-detected sites are included;
- AEST month and day boundaries are enforced;
- average site voltage is used;
- `ac_capacity_kw` is the standards rating basis;
- `s_99` is used only for the empirical apparent-limit symptom;
- the selected Volt-VAr capability profile is `review_corrected`;
- Stage 1 counterfactual availability and model qualification may differ.

Comparisons must be restricted to the same year and population before being
interpreted as methodological differences.

In [15]:
vv.compare_to_original(
    aq,
    database=DB,
    original="conformance_voltvar",
    target=VV_TARGET,
)

sites in conformance_voltvar_v2_flex_included:  16,148
sites in conformance_voltvar: 16,147
sites dropped:       0
flex-export sites (expected explanation for the drop): 539


(       n
 0  16148,
        n
 0  16147,
    n
 0  0,
      n
 0  539)

In [16]:
# Fleet-level headline comparison. Expect the same order of magnitude, not
# identical numbers.
aq(f"""
    SELECT 'v2_flex_included' AS tbl, year,
           round(sum(nonconformance_voltvar_sum), 0)  AS nonconf_kvar,
           round(sum(curtailment_voltvar_sum), 0)     AS curtail_kw,
           sum(total_count)                           AS intervals
    FROM {VV_TARGET} GROUP BY year
    ORDER BY year
""", database=DB)

,tbl,year,nonconf_kvar,curtail_kw,intervals
0,v2_flex_included,2024,225908026.0,1424.0,342883333
1,v2_flex_included,2025,187088805.0,671.0,305088775


## 8. Volt-VAr project-failure magnitude

`nonconformance_voltvar_red_sum` is the summed reactive-power shortfall for
the three categories treated as project failures:

- adverse response;
- inactive response;
- significant shortfall.

It excludes:

- near-conformant response;
- major reactive-power surplus.

The corresponding count is `nonconformance_voltvar_red_count`.

The sums are sums of kvar samples. Multiply by the five-minute interval
duration, `1/12 hour`, to obtain kvarh.

In [17]:
vvar_failure_totals = aq(f"""
    SELECT
        year,

        round(
            sum(nonconformance_voltvar_red_sum),
            3
        ) AS project_failure_kvar_sum,

        round(
            sum(nonconformance_voltvar_red_sum) / 12.0,
            3
        ) AS project_failure_kvarh,

        sum(nonconformance_voltvar_red_count)
            AS project_failure_intervals,

        round(sum(Q_adverse_sum), 3)
            AS adverse_kvar_sum,

        round(sum(Q_inactive_sum), 3)
            AS inactive_kvar_sum,

        round(sum(Q_significant_shortfall_sum), 3)
            AS significant_shortfall_kvar_sum,

        round(sum(Q_near_conformant_sum), 3)
            AS near_conformant_kvar_sum,

        round(sum(Q_major_surplus_sum), 3)
            AS major_surplus_kvar_sum

    FROM {VV_TARGET}

    GROUP BY year
    ORDER BY year
""", database=DB)

display(vvar_failure_totals)

,year,project_failure_kvar_sum,project_failure_kvarh,project_failure_intervals,adverse_kvar_sum,inactive_kvar_sum,significant_shortfall_kvar_sum,near_conformant_kvar_sum,major_surplus_kvar_sum
0,2024,2.090719e+08,1.742265e+07,153149547,7.188337e+07,1.236060e+08,1.358244e+07,871259.149,1.596491e+07
1,2025,1.724261e+08,1.436884e+07,128933687,5.978648e+07,9.860183e+07,1.403775e+07,807156.190,1.385559e+07


In [18]:
final_provenance = aq(f"""
    SELECT
        'volt_watt_basic' AS result,
        rating_basis,
        voltage_aggregation,
        flex_selection,
        CAST(NULL AS VARCHAR) AS empirical_limit_basis,
        CAST(NULL AS VARCHAR) AS capability_profile,
        count(*) AS n_rows,
        count(DISTINCT site_id) AS n_sites,
        min(year) AS first_year,
        max(year) AS last_year
    FROM {VW_BASIC_TARGET}
    GROUP BY
        rating_basis,
        voltage_aggregation,
        flex_selection

    UNION ALL

    SELECT
        'volt_watt_ghi',
        rating_basis,
        voltage_aggregation,
        flex_selection,
        CAST(NULL AS VARCHAR),
        CAST(NULL AS VARCHAR),
        count(*),
        count(DISTINCT site_id),
        min(year),
        max(year)
    FROM {VW_GHI_TARGET}
    GROUP BY
        rating_basis,
        voltage_aggregation,
        flex_selection

    UNION ALL

    SELECT
        'volt_var',
        rating_basis,
        voltage_aggregation,
        flex_selection,
        empirical_limit_basis,
        capability_profile,
        count(*),
        count(DISTINCT site_id),
        min(year),
        max(year)
    FROM {VV_TARGET}
    GROUP BY
        rating_basis,
        voltage_aggregation,
        flex_selection,
        empirical_limit_basis,
        capability_profile
""", database=DB)

display(final_provenance)

assert (final_provenance["n_rows"] > 0).all()
assert set(final_provenance["rating_basis"]) == {"ac_capacity_kw"}
assert set(final_provenance["voltage_aggregation"]) == {"avg"}
assert set(final_provenance["flex_selection"]) == {"include"}
assert set(final_provenance["first_year"]) == {2024}
assert set(final_provenance["last_year"]) == {2025}

vvar_provenance = final_provenance[
    final_provenance["result"] == "volt_var"
].iloc[0]

assert vvar_provenance["empirical_limit_basis"] == "s_99"
assert (
    vvar_provenance["capability_profile"]
    == VV_OPTIONS["capability_profile"]
)

print("Final Stage 2 provenance checks passed.")

,result,rating_basis,voltage_aggregation,flex_selection,empirical_limit_basis,capability_profile,n_rows,n_sites,first_year,last_year
0,volt_watt_basic,ac_capacity_kw,avg,include,<NA>,<NA>,15653733,16148,2024,2025
1,volt_watt_ghi,ac_capacity_kw,avg,include,<NA>,<NA>,15653733,16148,2024,2025
2,volt_var,ac_capacity_kw,avg,include,s_99,review_corrected,15653733,16148,2024,2025


Final Stage 2 provenance checks passed.
